# ?valuation finale avec r?entra?nement

Ce notebook s?lectionne d'abord un seuil sur les pas 25?34, puis r?entra?ne chaque mod?le sur tous les n?uds ?tiquet?s des pas 1?34. Pour le GCN et le Skip-GCN, l'?poque retenue pendant la s?lection est ensuite utilis?e comme nombre fixe d'?poques, sans validation pendant le r?entra?nement final.

Le vrai test (pas 35?49) n'est consult? qu'apr?s le r?entra?nement final. Il sert alors ? calculer les m?triques et le F1 par pas de temps ; il ne sert ? choisir ni seuil, ni ?poque, ni hyperparam?tre. Les n?uds `unknown` sont exclus de la perte et des m?triques.

In [ ]:
# Configuration de l'?valuation compl?te.
from pathlib import Path
import sys
import numpy as np

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src.data import load_data

SEEDS = [42]
GCN_EPOCHS = 600
MLP_EPOCHS = 200
MODELES = ["gcn", "skip-gcn"]
data = load_data()
counts = data.classes["class"].astype(str).value_counts().to_dict()
assert data.features.shape == (203_769, 167), data.features.shape
assert data.edges.shape == (234_355, 2), data.edges.shape
assert len(data.classes) == 203_769, len(data.classes)
assert counts.get("1", 0) == 4_545, counts
assert counts.get("2", 0) == 42_019, counts
assert counts.get("unknown", 0) == 157_205, counts
assert int(np.sum(data.y[data.test_mask] == 1)) == 1_083
assert not np.any(data.train_mask & data.val_mask)
assert not np.any(data.train_mask & data.test_mask)
assert not np.any(data.val_mask & data.test_mask)
assert not np.any((data.train_mask | data.val_mask | data.test_mask) & (data.y < 0))
print(f"Donn?es v?rifi?es : {len(data.y):,} n?uds, {len(data.edges):,} ar?tes")
print(f"Classes : illicites={counts['1']:,}, licites={counts['2']:,}, unknown={counts['unknown']:,}")
print(f"Support illicite du test : {int(np.sum(data.y[data.test_mask] == 1)):,}")

## S?lection puis r?entra?nement

La s?lection r?utilise le protocole existant sur entra?nement 1?24 et validation 25?34 pour fixer le seuil et, pour les GCN, la meilleure ?poque. Le r?entra?nement final ajuste un nouveau `StandardScaler` sur les lignes 1?34 seulement et entra?ne le mod?le sur les labels connus de cette p?riode. Le seuil s?lectionn? reste fig?.

In [ ]:
from time import perf_counter
import pandas as pd
from src.models import run_model_final

lignes_seed = []
lignes_pas = []
for nom in MODELES:
    for seed in SEEDS:
        debut = perf_counter()
        resultat = run_model_final(
            nom,
            data,
            seed,
            mlp_epochs=MLP_EPOCHS,
            gcn_epochs=GCN_EPOCHS,
        )
        duree = perf_counter() - debut
        ligne = {
            "modele": resultat["modele"],
            "seed": seed,
            "seuil_validation": resultat["threshold"],
            "epoques_utilisees": resultat["epochs_used"],
            "meilleure_epoque_selection": resultat["selection_best_epoch"],
            "duree_selection_secondes": resultat["selection_duration_seconds"],
            "duree_reentrainement_secondes": resultat["final_duration_seconds"],
            "duree_totale_secondes": duree,
            "support_test": resultat["support_test"],
            **resultat["test"],
        }
        lignes_seed.append(ligne)
        par_pas = resultat["f1_par_pas"].copy()
        par_pas.insert(0, "seed", seed)
        par_pas.insert(0, "modele", resultat["modele"])
        par_pas.insert(2, "seuil_validation", resultat["threshold"])
        lignes_pas.append(par_pas)
        print(f"{resultat['modele']} | seed={seed} | F1={resultat['test']['f1']:.4f} | {duree:.1f} s")

final_par_seed = pd.DataFrame(lignes_seed)
final_f1_par_pas = pd.concat(lignes_pas, ignore_index=True)
final_par_seed.head()

## R?sum? du test : moyenne et ?cart-type

Les m?triques rapport?es sont la pr?cision, le rappel, le F1 illicite et la PR-AUC. Le r?sum? est calcul? sur les seeds ; aucune accuracy n'est utilis?e.

In [ ]:
from src.metrics import aggregate_metrics

final_resume = aggregate_metrics(final_par_seed)
final_resume

## F1 du test par pas temporel

Le tableau et la courbe montrent la moyenne du F1 par pas sur les seeds. Le support indique le nombre de transactions illicites observ?es ? chaque pas.

In [ ]:
import matplotlib.pyplot as plt

pas_resume = (
    final_f1_par_pas.groupby(["modele", "time_step"], as_index=False)
    .agg(
        f1_moyen=("f1", "mean"),
        f1_ecart_type=("f1", "std"),
        support_illicite=("support_illicite", "first"),
    )
)
fig, ax = plt.subplots(figsize=(12, 6))
for modele, groupe in pas_resume.groupby("modele"):
    ax.plot(groupe["time_step"], groupe["f1_moyen"], marker="o", label=modele)
ax.set(title="F1 illicite du test apr?s r?entra?nement final", xlabel="Pas de temps", ylabel="F1")
ax.set_xticks(range(35, 50))
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
pas_resume

## Sauvegarde des r?sultats

Les trois fichiers s?parent le r?sum? agr?g?, les r?sultats par mod?le et seed, et les F1/supports par pas. `protocole_final.csv` est une copie des r?sultats par seed pour fournir un fichier principal ? plat, sans m?langer les types de lignes.

In [ ]:
repertoire_resultats = RACINE / "results"
repertoire_resultats.mkdir(parents=True, exist_ok=True)
final_resume.to_csv(repertoire_resultats / "final_resume.csv", index=False)
final_par_seed.to_csv(repertoire_resultats / "final_par_seed.csv", index=False)
final_f1_par_pas.to_csv(repertoire_resultats / "final_f1_par_pas.csv", index=False)
final_par_seed.to_csv(repertoire_resultats / "protocole_final.csv", index=False)
print(f"Fichiers enregistr?s dans : {repertoire_resultats}")